# 02 — Ingesta meteorológica (Open-Meteo)

Descarga clima diario 2017-2025 por distrito y lo cachea en `data/bronze/meteo/cache_meteo/` (local, ya no en Drive). Corre esta celda varias veces si Open-Meteo corta el batch por límite de rate.

In [1]:
import sys
from pathlib import Path

In [2]:
import pandas as pd
from denguekay.config.paths import REFERENCE
from denguekay.data.ingestion.openmeteo import download_pending_batch

distritos = pd.read_csv(REFERENCE / "distritos_piura_coords.csv")
print(len(distritos), "distritos")

65 distritos


In [3]:
import sys
print(sys.executable)

c:\Users\Rosa\RufoEsEternoElRegreso\Random\Dump\.venv\Scripts\python.exe


In [4]:
'''
Correr esta celda y luego resetear el kernel si es que sale algun relacionado con pyarrow. Si no, seguir con la siguiente celda.
'''

# %pip install --upgrade --force-reinstall pandas pyarrow

'\nCorrer esta celda y luego resetear el kernel si es que sale algun relacionado con pyarrow. Si no, seguir con la siguiente celda.\n'

In [5]:
# Corre esta celda repetidas veces hasta que diga 'Quedan: 0'
quedan = download_pending_batch(distritos)

Descargados: 65 | Pendientes: 0

En este batch: 0 | Quedan: 0
Listo: ya se puede pasar al armado de la data diaria (bronze -> silver).


## Estudio 06 · fase 1b · descargas aparte (no cambian lo de arriba)

Las celdas de arriba siguen descargando 2017-2025 con el modelo por defecto de Open-Meteo («Best match») en
`cache_meteo/`. Las de abajo escriben en **cachés separados**, así que no tocan esa descarga ni lo que lee
el notebook 03:

- **A · ERA5 fijo 2017-2025** → `data/bronze/meteo/cache_meteo_era5_2017_2025/`. Sirve para el notebook
  `research/06_model_improvement/notebooks/03_fase1b_consistencia_clima.ipynb`. Se corre primero.
- **B · historia 2000-2016** → `data/bronze/meteo/cache_meteo_2000_2016/`. Se corre **después** de que Rosa
  decida con qué modelo, a la vista de esa comparación.

Antes de correr A, confirmen el identificador del modelo: en https://open-meteo.com/en/docs/historical-weather-api
elijan ERA5 en «Reanalysis models» y miren el valor de `models=` en la URL que genera la página. Si no es
`era5`, cámbienlo en `MODELO_FIJO`. Si el API responde con columnas renombradas, `fetch_district` se detiene.
Igual que arriba: corran cada celda varias veces hasta que diga «Quedan: 0».


In [ ]:
from denguekay.config.paths import CACHE_METEO_2000_2016, CACHE_METEO_ERA5_2017_2025

MODELO_FIJO = "era5"  # ERA5, 0,25°, desde 1940; ERA5-Land no trae radiación solar según la documentación

# A · Corre esta celda repetidas veces hasta que diga 'Quedan: 0'
quedan_era5 = download_pending_batch(
    distritos, models=MODELO_FIJO, cache_dir=CACHE_METEO_ERA5_2017_2025,
)


In [ ]:
# B · Solo después de la decisión de Rosa. "best_match" = sin fijar modelo (como 2017-2025).
MODELO_HISTORIA = "POR_DECIDIR"
if MODELO_HISTORIA == "POR_DECIDIR":
    raise RuntimeError("Falta la decisión de Rosa sobre el modelo de la historia 2000-2016.")

quedan_historia = download_pending_batch(
    distritos, start_date="2000-01-01", end_date="2016-12-31",
    models=None if MODELO_HISTORIA == "best_match" else MODELO_HISTORIA,
    cache_dir=CACHE_METEO_2000_2016,
)
